##  Basic Library imports

In [15]:
import os
import pandas as pd 
import numpy as np
from tqdm import tqdm

##  Read Dataset

In [4]:
DATASET_FOLDER = '../dataset/'
train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
test = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))
sample_test = pd.read_csv(os.path.join(DATASET_FOLDER, 'sample_test.csv'))
sample_test_out = pd.read_csv(os.path.join(DATASET_FOLDER, 'sample_test_out.csv'))

In [ ]:
from utils import download_images
download_images(train['image_link'], '../train_images')

In [ ]:
# Retry missing downloads in the notebook
import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from tqdm import tqdm
import csv

TRAIN_FOLDER = '../train_images'
os.makedirs(TRAIN_FOLDER, exist_ok=True)

# assumes `train` DataFrame is already loaded in the notebook
links = train['image_link'].astype(str).tolist()
print(len(links))
def filename_from_link(link):
    return Path(link).name

# Build list of links that are not already downloaded
missing_links = []
for link in links:
    if not isinstance(link, str) or not link.strip():
        continue
    fname = filename_from_link(link)
    dest = Path(TRAIN_FOLDER) / fname
    if not dest.exists():
        missing_links.append(link)

print(f"To download: {len(missing_links)} missing images")

def download_one(link, folder, retries=3, timeout=10):
    if not isinstance(link, str) or not link.strip():
        return False
    fname = filename_from_link(link)
    dest_path = Path(folder) / fname
    if dest_path.exists():
        return True
    tmp = dest_path.with_suffix(dest_path.suffix + '.part')
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(link, stream=True, timeout=timeout)
            if resp.status_code == 200:
                with open(tmp, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp, dest_path)
                return True
            else:
                # non-200 response
                # optionally log: 
                print(f"Bad status {resp.status_code} for {link}")
                pass
        except Exception:
            # network error or write error
            pass
        # exponential backoff
        time.sleep(2 ** (attempt - 1))
    # cleanup partial file if exists
    try:
        if tmp.exists():
            tmp.unlink()
    except Exception:
        pass
    return False

# adjust workers if needed; keep conservative value for notebook
max_workers = min(32, (os.cpu_count() or 1) * 5, 20)

failed = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    future_to_link = {ex.submit(download_one, link, TRAIN_FOLDER, 3): link for link in missing_links}
    for fut in tqdm(as_completed(future_to_link), total=len(future_to_link), desc="Retry downloads"):
        link = future_to_link[fut]
        try:
            ok = fut.result()
            if not ok:
                failed.append(link)
        except Exception:
            failed.append(link)

print(f"Done. Success: {len(missing_links) - len(failed)}, Failed: {len(failed)}")

# Save failed links for another retry later
if failed:
    failed_csv = Path('failed_links_retry.csv')
    with open(failed_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['image_link'])
        for l in failed:
            writer.writerow([l])
    print(f"Saved failed links to {failed_csv}")

75000
To download: 1 missing images


Retry downloads:   0%|          | 0/1 [00:00<?, ?it/s]

Bad status 404 for https://m.media-amazon.com/images/I/51mjZYDYjyL.jpg
Bad status 404 for https://m.media-amazon.com/images/I/51mjZYDYjyL.jpg
Bad status 404 for https://m.media-amazon.com/images/I/51mjZYDYjyL.jpg


Retry downloads: 100%|██████████| 1/1 [00:07<00:00,  7.31s/it]

Done. Success: 0, Failed: 1
Saved failed links to failed_links_retry.csv


In [17]:
from utils import download_images
download_images(test['image_link'], '../test_images')

[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2548)


[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2548)


Remote end closed connection without response


[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2548)


[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2548)
[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2548)
[SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:2548)
[SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:2548)


...

Remote end closed connection without response


Remote end closed connection without response
Remote end closed connection without response
Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/71Wu0z0qUVL.jpg
Remote end closed connection without response

Remote end closed connection without response
Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/71bsxQULMaL.jpg
Remote end closed connection without response

<urlopen error [SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:997)>
Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/61pf6EW6GPL.jpg
Remote end closed connection without response

Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/614vvowLqAL.jpg
Remote end closed connection without response

Remote end closed connection without response
<url

Remote end closed connection without response
Remote end closed connection without response
<urlopen error [SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:997)>
Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/71Ucmn9f66L.jpg
Remote end closed connection without response

Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/81oV2OoV+rL.jpg
Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/61I67gfZOOL.jpg
Remote end closed connection without response
Remote end closed connection without response


Remote end closed connection without response
Remote end closed connection without responseWarning: Not able to download - https://m.media-amazon.com/images/I/81iJmZzzBnL.jpg
Remote end closed connection without responseWarning: Not able to download - https://m.media-amaz

Process ForkPoolWorker-1903:
 53%|█████▎    | 39623/75000 [2:47:14<2:29:19,  3.95it/s] Process ForkPoolWorker-1905:
Process ForkPoolWorker-1879:
Process ForkPoolWorker-1913:
Process ForkPoolWorker-1915:
Process ForkPoolWorker-1904:
Process ForkPoolWorker-1865:
Process ForkPoolWorker-1894:
Process ForkPoolWorker-1871:
Process ForkPoolWorker-1912:
Process ForkPoolWorker-1891:
Process ForkPoolWorker-1914:
Process ForkPoolWorker-1847:
Process ForkPoolWorker-1906:
Process ForkPoolWorker-1851:
Process ForkPoolWorker-1885:
Process ForkPoolWorker-1908:
Process ForkPoolWorker-1880:
Process ForkPoolWorker-1883:
Process ForkPoolWorker-1882:
Process ForkPoolWorker-1837:
Process ForkPoolWorker-1890:
Process ForkPoolWorker-1896:
Traceback (most recent call last):
Process ForkPoolWorker-1876:
Process ForkPoolWorker-1823:
Process ForkPoolWorker-1878:
Process ForkPoolWorker-1874:
Process ForkPoolWorker-1842:
Process ForkPoolWorker-1848:
Process ForkPoolWorker-1901:

Copy/Download test images:   0%|    

KeyboardInterrupt: 

In [18]:
# Retry missing downloads in the notebook
import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from tqdm import tqdm
import csv

test_FOLDER = '../test_images'
os.makedirs(test_FOLDER, exist_ok=True)

# assumes `test` DataFrame is already loaded in the notebook
links = test['image_link'].astype(str).tolist()
print(len(links))
def filename_from_link(link):
    return Path(link).name

# Build list of links that are not already downloaded
missing_links = []
for link in links:
    if not isinstance(link, str) or not link.strip():
        continue
    fname = filename_from_link(link)
    dest = Path(test_FOLDER) / fname
    if not dest.exists():
        missing_links.append(link)

print(f"To download: {len(missing_links)} missing images")

def download_one(link, folder, retries=3, timeout=10):
    if not isinstance(link, str) or not link.strip():
        return False
    fname = filename_from_link(link)
    dest_path = Path(folder) / fname
    if dest_path.exists():
        return True
    tmp = dest_path.with_suffix(dest_path.suffix + '.part')
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(link, stream=True, timeout=timeout)
            if resp.status_code == 200:
                with open(tmp, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp, dest_path)
                return True
            else:
                # non-200 response
                # optionally log: 
                print(f"Bad status {resp.status_code} for {link}")
                pass
        except Exception:
            # network error or write error
            pass
        # exponential backoff
        time.sleep(2 ** (attempt - 1))
    # cleanup partial file if exists
    try:
        if tmp.exists():
            tmp.unlink()
    except Exception:
        pass
    return False

# adjust workers if needed; keep conservative value for notebook
max_workers = min(32, (os.cpu_count() or 1) * 5, 20)

failed = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    future_to_link = {ex.submit(download_one, link, test_FOLDER, 3): link for link in missing_links}
    for fut in tqdm(as_completed(future_to_link), total=len(future_to_link), desc="Retry downloads"):
        link = future_to_link[fut]
        try:
            ok = fut.result()
            if not ok:
                failed.append(link)
        except Exception:
            failed.append(link)

print(f"Done. Success: {len(missing_links) - len(failed)}, Failed: {len(failed)}")

# Save failed links for another retry later
if failed:
    failed_csv = Path('failed_links_retry.csv')
    with open(failed_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['image_link'])
        for l in failed:
            writer.writerow([l])
    print(f"Saved failed links to {failed_csv}")

75000
To download: 130 missing images


Retry downloads:  55%|█████▍    | 71/130 [00:13<00:11,  4.97it/s]

Bad status 404 for https://m.media-amazon.com/images/I/813CjSgHj0S.jpg


Retry downloads:  66%|██████▌   | 86/130 [00:17<00:13,  3.34it/s]

Bad status 404 for https://m.media-amazon.com/images/I/813CjSgHj0S.jpg


Retry downloads:  85%|████████▌ | 111/130 [00:20<00:02,  9.33it/s]

Bad status 404 for https://m.media-amazon.com/images/I/813CjSgHj0S.jpg


Retry downloads: 100%|██████████| 130/130 [00:24<00:00,  5.30it/s]

Done. Success: 129, Failed: 1
Saved failed links to failed_links_retry.csv


In [ ]:
assert len(os.listdir('../train_images')) == 75000